# audio_controls

> Audio playback controls for the alignment card stack: auto-navigate toggle

In [ ]:
#| default_exp components.audio_controls

In [ ]:
#| export
from typing import Any, List

from fasthtml.common import Div, Span, Input, Label, Select, Option

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_input.toggle import toggle, toggle_sizes
from cjm_fasthtml_daisyui.components.data_input.select import select, select_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.typography import font_size
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

## HTML IDs

HTML ID constants for alignment audio control elements.

In [ ]:
#| export
class AlignAudioControlIds:
    """HTML ID constants for alignment audio control elements."""

    SPEED_SELECT = "sd-align-speed-select"
    AUTO_NAV_TOGGLE = "sd-align-auto-nav-toggle"
    AUDIO_CONTROLS = "sd-align-audio-controls"

## Speed Selector

Dropdown for adjusting playback speed. Client-side `onchange` calls `window.setAlignSpeed(...)` (exposed by `cjm-fasthtml-web-audio` when `enable_speed=True`), which is pitch-preserving via the SoundTouch worklet. If a `change_url` is provided, the select also POSTs the new speed so the server can persist it to `AlignmentStepState.playback_speed`.

In [ ]:
#| export
# Available playback speeds (value, label)
PLAYBACK_SPEEDS: List[tuple] = [
    (0.5, "0.5x"),
    (0.75, "0.75x"),
    (1.0, "1x"),
    (1.25, "1.25x"),
    (1.5, "1.5x"),
    (2.0, "2x"),
    (2.5, "2.5x"),
    (3.0, "3x"),
]

def render_align_speed_selector(
    current_speed:float=1.0,  # Current playback speed
    change_url:str="",  # URL to POST speed changes to (for server persistence)
) -> Any:  # Speed selector component
    """Render playback speed selector dropdown for alignment audio."""
    options = [
        Option(
            label,
            value=str(speed),
            selected=(speed == current_speed),
        )
        for speed, label in PLAYBACK_SPEEDS
    ]

    # Build HTMX attributes if URL provided
    htmx_attrs = {}
    if change_url:
        htmx_attrs = {
            "hx_post": change_url,
            "hx_trigger": "change",
            "hx_swap": "none",
        }

    # Client-side JS to update playback speed immediately via web audio library
    onchange_js = "if(window.setAlignSpeed) window.setAlignSpeed(parseFloat(this.value));"

    return Div(
        Span(
            "Speed:",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70), m.r(2))
        ),
        Select(
            *options,
            id=AlignAudioControlIds.SPEED_SELECT,
            name="speed",
            cls=combine_classes(select, select_sizes.sm),
            onchange=onchange_js,
            **htmx_attrs,
        ),
        cls=combine_classes(flex_display, items.center)
    )

## Auto-Navigate Toggle

Toggle switch to enable automatic navigation to the next VAD chunk
when audio playback of the current chunk completes.

In [ ]:
#| export
# CSS classes for toggle state coloring
_TOGGLE_BG_OFF = str(bg_dui.error)    # Red when auto-play disabled
_TOGGLE_BG_ON = str(bg_dui.success)   # Green when auto-play enabled

def _toggle_color_js(toggle_id:str) -> str:  # JS snippet to sync toggle color classes
    """Generate JS to swap bg-error/bg-success on the toggle based on checked state."""
    return (
        f"var _t=document.getElementById('{toggle_id}');"
        f"if(_t){{_t.classList.remove('{_TOGGLE_BG_OFF}','{_TOGGLE_BG_ON}');"
        f"_t.classList.add(_t.checked?'{_TOGGLE_BG_ON}':'{_TOGGLE_BG_OFF}');}}"
    )

def render_align_auto_navigate_toggle(
    enabled:bool=False,  # Whether auto-navigate is enabled
) -> Any:  # Auto-navigate toggle component
    """Render auto-navigate toggle switch for alignment audio (client-side only)."""
    toggle_id = AlignAudioControlIds.AUTO_NAV_TOGGLE
    color_js = _toggle_color_js(toggle_id)
    onchange_js = f"if(window.setAlignAutoNavigate) window.setAlignAutoNavigate(this.checked);{color_js}"

    # Only pass checked when enabled (FastHTML renders checked=False as an attribute)
    check_attr = {"checked": True} if enabled else {}
    color_cls = _TOGGLE_BG_ON if enabled else _TOGGLE_BG_OFF

    return Div(
        Label(
            Span(
                "Auto-play:",
                cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70), m.r(2))
            ),
            Input(
                type="checkbox",
                id=toggle_id,
                name="auto_navigate",
                cls=combine_classes(toggle, toggle_sizes.sm, color_cls),
                onchange=onchange_js,
                **check_attr,
            ),
            cls=combine_classes(flex_display, items.center)
        ),
        cls=combine_classes(flex_display, items.center)
    )

## Audio Controls Container

Wraps the toggle in an OOB-targetable container for chrome switching.

In [ ]:
#| export
def render_align_audio_controls(
    current_speed:float=1.0,  # Current playback speed
    auto_navigate:bool=False,  # Whether auto-navigate is enabled
    speed_url:str="",  # URL for speed changes (server persistence)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Combined audio controls component
    """Render alignment audio controls (speed selector + auto-navigate toggle)."""
    return Div(
        render_align_speed_selector(current_speed, speed_url),
        render_align_auto_navigate_toggle(auto_navigate),
        id=AlignAudioControlIds.AUDIO_CONTROLS,
        cls=combine_classes(flex_display, items.center, gap(4)),
        hx_swap_oob="true" if oob else None,
    )

## Tests

In [ ]:
from fasthtml.common import to_xml
import re

def _get_toggle_classes(html):
    """Extract class attribute from the toggle input element."""
    m = re.search(r'id="sd-align-auto-nav-toggle"[^>]*class="([^"]*)"', html)
    return m.group(1) if m else ""

# Test enabled toggle — green background, checked
html = to_xml(render_align_auto_navigate_toggle(enabled=True))
assert ' checked' in html
assert 'sd-align-auto-nav-toggle' in html
assert 'setAlignAutoNavigate' in html
assert 'hx-post' not in html  # No server persistence
toggle_cls = _get_toggle_classes(html)
assert 'bg-success' in toggle_cls
assert 'bg-error' not in toggle_cls

# Test disabled state — red background, no checked
html_off = to_xml(render_align_auto_navigate_toggle(enabled=False))
assert ' checked' not in html_off
toggle_cls_off = _get_toggle_classes(html_off)
assert 'bg-error' in toggle_cls_off
assert 'bg-success' not in toggle_cls_off

# Test color JS is in onchange (classList swap logic)
assert 'classList.remove' in html
assert 'classList.add' in html

# Speed selector tests
speed_html = to_xml(render_align_speed_selector(current_speed=1.5))
assert 'sd-align-speed-select' in speed_html
assert 'setAlignSpeed' in speed_html
# All 8 PLAYBACK_SPEEDS options rendered
for speed, label in PLAYBACK_SPEEDS:
    assert f'value="{speed}"' in speed_html
    assert f'>{label}<' in speed_html
# 1.5x is selected
assert re.search(r'<option[^>]*value="1.5"[^>]*selected', speed_html)
# No server URL → no HTMX attributes
assert 'hx-post' not in speed_html

# Speed selector with server persistence URL
speed_html_url = to_xml(render_align_speed_selector(current_speed=2.0, change_url="/speed"))
assert 'hx-post="/speed"' in speed_html_url
assert 'hx-swap="none"' in speed_html_url

# Container includes BOTH speed selector + auto-nav toggle
html_combined = to_xml(render_align_audio_controls(current_speed=2.0, auto_navigate=True))
assert 'sd-align-speed-select' in html_combined
assert 'sd-align-auto-nav-toggle' in html_combined
assert 'sd-align-audio-controls' in html_combined

# Test container with OOB
html_oob = to_xml(render_align_audio_controls(auto_navigate=True, oob=True))
assert 'hx-swap-oob' in html_oob
assert 'sd-align-audio-controls' in html_oob

print('Alignment audio controls tests passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()